# 根据输入提示生成物体掩膜

## 安装依赖

In [2]:
%pip install segment-geospatial
%pip install leafmap
%pip install localtileserver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.2/157.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.0/102.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.2/669.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 

In [3]:
import leafmap

from samgeo import SamGeo
from samgeo.common import tms_to_geotiff

## 创建交互地图

In [4]:
m = leafmap.Map(center=[37.6412, -122.1353], zoom=15, height="800px")
m.add_basemap("SATELLITE")
m

Map(center=[37.6412, -122.1353], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', '…

## 下载影像
平移和缩放地图以选定感兴趣的区域。可以使用绘图工具在地图上绘制多边形或矩形。

In [5]:
if m.user_roi is not None:
    bbox = m.user_roi_bounds()
else:
    bbox = [-122.1497, 37.6311, -122.1203, 37.6458]

In [6]:
image = "satellite.tif"
tms_to_geotiff(output=image, bbox=bbox, zoom=16, source="Satellite", overwrite=True)

Downloaded image 01/30
Downloaded image 02/30
Downloaded image 03/30
Downloaded image 04/30
Downloaded image 05/30
Downloaded image 06/30
Downloaded image 07/30
Downloaded image 08/30
Downloaded image 09/30
Downloaded image 10/30
Downloaded image 11/30
Downloaded image 12/30
Downloaded image 13/30
Downloaded image 14/30
Downloaded image 15/30
Downloaded image 16/30
Downloaded image 17/30
Downloaded image 18/30
Downloaded image 19/30
Downloaded image 20/30
Downloaded image 21/30
Downloaded image 22/30
Downloaded image 23/30
Downloaded image 24/30
Downloaded image 25/30
Downloaded image 26/30
Downloaded image 27/30
Downloaded image 28/30
Downloaded image 29/30
Downloaded image 30/30
Saving GeoTIFF. Please wait...
Image saved to satellite.tif


取消注释并运行下面的代码，即可使用自己的图片。

In [ ]:
# image = '/path/to/your/own/image.tif'

在地图上显示下载后的图片。

In [7]:
m.layers[-1].visible = False
m.add_raster(image, layer_name="Image")
m

Map(bottom=3246711.0, center=[37.638450000000006, -122.13499999999999], controls=(ZoomControl(options=['positi…

##初始化 SAM 类

指定模型检查点的文件路径。如果未指定，则模型将被下载到工作目录中。

In [8]:
sam = SamGeo(
    model_type="vit_h",
    automatic=False,
    sam_kwargs=None,
)

Model checkpoint for vit_h not found.


Downloading...
From: https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
To: /root/.cache/torch/hub/checkpoints/sam_vit_h_4b8939.pth
100%|██████████| 2.56G/2.56G [00:15<00:00, 167MB/s] 


定义需要进行分割的影像

In [9]:
sam.set_image(image)

## 基于输入点的图像分割

可以使用单个点来划分某个对象。该点可以以(x, y)的形式来表示，比如(col, row)或(lon, lat)。此外，这些点也可以作为向量数据集的文件路径来指定。对于非(col, row)格式的输入点，需要指定 point_crs 参数，该参数会自动将点转换为图像中的列和行坐标。

In [10]:
point_coords = [[-122.1419, 37.6383]]
sam.predict(point_coords, point_labels=1, point_crs="EPSG:4326", output="mask1.tif")
m.add_raster("mask1.tif", layer_name="Mask1", nodata=0, cmap="Blues", opacity=1)
m

Map(bottom=1623596.0, center=[37.638450000000006, -122.13499999999999], controls=(ZoomControl(options=['positi…

输入多个点：

In [11]:
point_coords = [[-122.1464, 37.6431], [-122.1449, 37.6415], [-122.1451, 37.6395]]
sam.predict(point_coords, point_labels=1, point_crs="EPSG:4326", output="mask2.tif")
m.add_raster("mask2.tif", layer_name="Mask2", nodata=0, cmap="Greens", opacity=1)
m

Map(bottom=1623596.0, center=[37.638450000000006, -122.13499999999999], controls=(ZoomControl(options=['positi…

## 交互式分割

显示交互式地图，使用标记工具在地图上绘制点。然后点击 Segment 按钮来对这些点进行分组处理。处理结果会自动添加到地图上。点击 Reset 按钮则可以清除所有的点和处理结果。

In [12]:
m = sam.show_map()
m

Map(center=[37.638450000000006, -122.13499999999999], controls=(ZoomControl(options=['position', 'zoom_in_text…

![](https://i.imgur.com/2Nyg9uW.gif)